# Salting: Fixing Data Skew in Joins and Aggregations

**What this notebook does**

* Builds a uniform table (`df_uniform`, values 0 to 999,999) and a heavily skewed table (`df_skew`, 999,990 rows with `value = 0` and only 30 rows for values 1 to 3).
* Joins them and shows that almost all joined rows land in a single shuffle partition.
* Applies **salting** to spread the hot key across partitions, for both a join and an aggregation.

**Core concept: salting**

In a shuffle, Spark sends every row with the same key to the same partition (`hash(key) % numPartitions`). A hot key therefore becomes one huge task. Salting adds a random number (the *salt*, 0 to N-1) to the key on the skewed side, so one hot key becomes N different keys `(key, salt)` that hash to different partitions:

* **Join:** the skewed side gets a random salt per row; the other side is **replicated N times** (one copy per salt value via `explode`) so every `(key, salt)` still finds its match. Cost: the replicated side grows N times.
* **Aggregation:** aggregate on `(key, salt)` first (partial results, spread across tasks), then aggregate again on `key` alone (small, because there are at most N rows per key).

**Databricks conversion notes**

* Run on an **interactive (all-purpose) cluster** such as *Sandani Personal Compute*. Classic clusters let you disable AQE and broadcast joins, which is required to see raw skew; serverless does not allow these settings.
* The local `SparkSession.builder.master("local[*]")` setup was removed: Databricks provides `spark`. `spark.stop()` was removed because the session is managed by the cluster.
* `spark.createDataFrame([...python list...])` was replaced by `spark.range(...)`: the data is generated on the executors instead of being built in Python on the driver and uploaded (much faster under Spark Connect in standard access mode).
* No external data source is needed; all data is generated in the notebook.

## 1. Setup

In [0]:
# Jupyter-only InteractiveShell setting removed; Databricks renders cell output natively.
print(f"Spark version: {spark.version}")

In [0]:
from pyspark.sql.types import *
import pyspark.sql.functions as F
# SparkSession is not imported: Databricks provides a ready-to-use `spark` object.

In [0]:
# Original (local Jupyter): spark = SparkSession.builder.master("local[*]").getOrCreate()
# On Databricks the session already exists.

# df_uniform is only ~4 MB, below the 10 MB broadcast threshold. Databricks would broadcast it and
# skip the shuffle entirely, hiding the skew. Force a shuffle (sort-merge) join instead.
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)
print("autoBroadcastJoinThreshold =", spark.conf.get("spark.sql.autoBroadcastJoinThreshold"))

In [0]:
# 3 shuffle partitions keep the output easy to read: every join/groupBy produces 3 partitions.
spark.conf.set("spark.sql.shuffle.partitions", "3")

# Disable Adaptive Query Execution: AQE would coalesce partitions and split skewed join partitions
# automatically, hiding the problem that salting solves manually.
spark.conf.set("spark.sql.adaptive.enabled", "false")

print("shuffle.partitions =", spark.conf.get("spark.sql.shuffle.partitions"))
print("adaptive.enabled   =", spark.conf.get("spark.sql.adaptive.enabled"))

## 2. Simulating a Skewed Join

**What:** Create two DataFrames with a single integer column `value`:

* `df_uniform`: one row per value from 0 to 999,999, spread evenly across partitions.
* `df_skew`: 999,990 rows with `value = 0`, 15 rows with `1`, 10 with `2`, 5 with `3`. Each piece is forced into one partition with `repartition(1)` and combined with `union` (no shuffle), so partition 0 holds almost all rows.

**Core concept:** `F.spark_partition_id()` tags each row with its partition id; grouping by it measures how evenly data is spread. The uniform DataFrame shows near-equal counts per partition (the number of partitions depends on the cluster's cores); the skewed one shows 999,990 vs 15 / 10 / 5.

In [0]:
# Original: spark.createDataFrame([i for i in range(1000000)], IntegerType())
# spark.range generates the rows on the executors; cast to int and name the column "value" to match.
df_uniform = spark.range(1000000).select(F.col("id").cast("int").alias("value"))
display(df_uniform.limit(5))

In [0]:
(
    df_uniform
    .withColumn("partition", F.spark_partition_id())
    .groupBy("partition")
    .count()
    .orderBy("partition")
    .show(15, False)
)

In [0]:
# Original: spark.createDataFrame([0] * 999990, IntegerType()) etc.
# spark.range(n) + a constant column produces n rows with the same value, without building a Python list.
def constant_rows(value, n):
    return spark.range(n).select(F.lit(value).cast("int").alias("value")).repartition(1)

df0 = constant_rows(0, 999990)
df1 = constant_rows(1, 15)
df2 = constant_rows(2, 10)
df3 = constant_rows(3, 5)
df_skew = df0.union(df1).union(df2).union(df3)
display(df_skew.limit(5))

In [0]:
(
    df_skew
    .withColumn("partition", F.spark_partition_id())
    .groupBy("partition")
    .count()
    .orderBy("partition")
    .show()
)

**What:** Inner join `df_skew` with `df_uniform` on `value`, then count joined rows per partition.

**Core concept:** With broadcast and AQE disabled this is a **sort-merge join**: both sides are shuffled by `hash(value) % 3`. All 999,990 rows with `value = 0` go to the same partition, so one task produces nearly every joined row while the other two finish almost instantly. Check the join stage in **Spark UI > Stages**: one task's duration and shuffle-read size dwarf the others.

In [0]:
df_joined_c1 = df_skew.join(df_uniform, "value", "inner")

In [0]:
(
    df_joined_c1
    .withColumn("partition", F.spark_partition_id())
    .groupBy("partition")
    .count()
    .orderBy("partition")
    .show()#5, False)
)

## 3. Salting in Joins

**What:** Choose the number of salt values `SALT_NUMBER` (here equal to the 3 shuffle partitions) and add a random `salt` column (0, 1 or 2) to every row of `df_skew`.

**Core concept:** `F.rand()` returns a random number in [0, 1); multiplying by `SALT_NUMBER` and casting to `int` gives an evenly distributed salt. The 999,990 rows of `value = 0` are now split into about 333,330 rows each for `(0, 0)`, `(0, 1)` and `(0, 2)`, three different shuffle keys.

**Choosing the salt number:** a larger salt spreads the hot key over more tasks but multiplies the size of the replicated side by the same factor. Pick the smallest number that brings the hot partition close to the average size.

In [0]:
SALT_NUMBER = int(spark.conf.get("spark.sql.shuffle.partitions"))
print("SALT_NUMBER =", SALT_NUMBER)

In [0]:
df_skew = df_skew.withColumn("salt", (F.rand() * SALT_NUMBER).cast("int"))

In [0]:
display(df_skew.limit(10))

**What:** Give every row of `df_uniform` all salt values: build an array `[0, 1, 2]` and `explode` it into one row per salt. Then join on `["value", "salt"]`.

**Core concept:** The skewed side picked its salt at random, so we do not know which salt a given row has. Replicating the other side with every possible salt guarantees each `(value, salt)` finds its match, so the join result is identical to the unsalted join. `df_uniform` grows from 1M to 3M rows, which is the price paid for spreading the hot key.

**Expected result:** rows for `value = 0` now appear in several partitions (about 333K each) instead of one. Because partitions are assigned by `hash(value, salt) % 3`, two salts can occasionally hash to the same partition, so the split is not always perfectly even; a larger salt and partition count smooths this out.

In [0]:
# Every row is copied SALT_NUMBER times (salt = 0..N-1) so each (value, salt) on the skewed side has a match.
df_uniform = (
    df_uniform
    .withColumn("salt_values", F.array([F.lit(i) for i in range(SALT_NUMBER)]))
    .withColumn("salt", F.explode(F.col("salt_values")))
)

In [0]:
display(df_uniform.limit(10))

In [0]:
# Join on the composite key: rows are now shuffled by hash(value, salt) instead of hash(value).
df_joined = df_skew.join(df_uniform, ["value", "salt"], "inner")

In [0]:
(
    df_joined
    .withColumn("partition", F.spark_partition_id())
    .groupBy("value", "partition")
    .count()
    .orderBy("value", "partition")
    .show()
)

## 4. Salting in Aggregations

**What:** Count rows per `value`, first directly, then with a two-step salted aggregation.

**Core concept:** A plain `groupBy("value")` sends all 999,990 rows of `value = 0` to one task for the final count. The salted version:

1. `groupBy("value", "salt")` produces partial counts, with the hot key split across `SALT_NUMBER` tasks.
2. `groupBy("value")` with `sum(count)` combines at most `SALT_NUMBER` partial rows per key, which is cheap.

Both versions return the same counts; only the work distribution changes. This works for any aggregate that can be combined from partial results (count, sum, min, max). `avg` must be rebuilt from sum and count, and exact `countDistinct` cannot be split this way.

**Note:** Spark already does a partial (map-side) aggregation before the shuffle, so plain `count` is less affected by skew than a join. Salting matters most for aggregations that cannot be partially combined, such as `collect_list`.

In [0]:
df_skew.groupBy("value").count().show()

In [0]:
(
    df_skew
    .withColumn("salt", (F.rand() * SALT_NUMBER).cast("int"))
    .groupBy("value", "salt")
    .agg(F.count("value").alias("count"))
    .groupBy("value")
    .agg(F.sum("count").alias("count"))
    .show()
)

In [0]:
# Original: spark.stop()
# Do not stop the session on Databricks: it is managed by the cluster and shared by the notebook.
# Optionally restore defaults for later work in this session:
# spark.conf.unset("spark.sql.shuffle.partitions")
# spark.conf.unset("spark.sql.adaptive.enabled")
# spark.conf.unset("spark.sql.autoBroadcastJoinThreshold")

## Summary

| | Without salting | With salting |
| --- | --- | --- |
| Shuffle key | `value` | `(value, salt)` |
| Hot key `0` | 1 partition, ~1M rows | spread over up to `SALT_NUMBER` partitions |
| Extra cost | none | other join side replicated `SALT_NUMBER` times; two-step aggregation |

**When to use salting on Databricks:** AQE's skew-join handling (`spark.sql.adaptive.skewJoin.enabled`, on by default) splits skewed join partitions automatically, and broadcast joins avoid the shuffle when one side is small. Use manual salting when those are not enough: extremely hot keys, skewed aggregations (AQE only handles joins), or both join sides too large to broadcast.

**Previous:** `2_skew_dataset_simulation`, `3_solving_data_skew_aqe_broadcast`.